# Contract Document OCR & Structured Extraction

This notebook is based on the original contract OCR experiment in the project archive. It sends a document image to a Hugging Face-hosted vision-language model and requests structured Arabic JSON extraction.

**Security:** provide your Hugging Face token through the `HUGGINGFACEHUB_API_TOKEN` environment variable. No API key is stored in this notebook.

In [ ]:
import os
import json
import base64
import mimetypes
from pathlib import Path

from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError("Set HUGGINGFACEHUB_API_TOKEN before running this notebook.")

client = InferenceClient(api_key=HF_TOKEN)
MODEL_ID = "thinkingmachines/Inkling:together"

## 1. Load and encode a document image

Set `IMAGE_PATH` to a contract image that you are authorized to process.

In [ ]:
IMAGE_PATH = "sample_contract.png"

def encode_image(image_path: str) -> tuple[str, str]:
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    mime_type, _ = mimetypes.guess_type(path.name)
    if not mime_type or not mime_type.startswith("image/"):
        mime_type = "image/png"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return encoded, mime_type

image_base64, image_mime = encode_image(IMAGE_PATH)
print(f"Loaded {IMAGE_PATH} ({image_mime})")

## 2. Define extraction instructions

The prompt asks the model to preserve empty fields and avoid inventing unreadable values.

In [ ]:
SYSTEM_PROMPT = """
أنت نموذج ذكاء اصطناعي متخصص في OCR واستخراج بيانات العقود.

حلل الصورة وأجب فقط بصيغة JSON.

إذا كانت الصورة ليست عقدًا أو مستندًا قانونيًا، أعد:
{
  "status": "not_contract",
  "message": "الصورة المدخلة ليست عقدًا."
}

إذا كانت الصورة عقدًا:
- استخرج الحقول الموجودة، واجعل اسم الحقل هو المفتاح والإجابة هي القيمة.
- إذا كان الحقل فارغًا (نقاط فقط أو مساحة فارغة) أعد القيمة كـ "".
- لا تحذف الحقول الفارغة.
- لا تخمن أي معلومات غير واضحة.
- أعد JSON فقط بدون أي شرح.
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "استخرج بيانات العقد من الصورة."},
            {
                "type": "image_url",
                "image_url": {"url": f"data:{image_mime};base64,{image_base64}"}
            }
        ]
    }
]

## 3. Run inference

In [ ]:
completion = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=8192,
    temperature=0
)

model_output = completion.choices[0].message.content
print(model_output)

## 4. Parse and save JSON

If the model wraps its response in a Markdown code fence, remove the fence before parsing.

In [ ]:
def parse_json_response(model_output: str) -> dict:
    text = model_output.strip()

    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError("Model output is not valid JSON. Inspect model_output.") from exc

contract_data = parse_json_response(model_output)

OUTPUT_PATH = "contract_result.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as output_file:
    json.dump(contract_data, output_file, ensure_ascii=False, indent=2)

print(json.dumps(contract_data, ensure_ascii=False, indent=2))
print(f"Saved to: {OUTPUT_PATH}")

## Limitations

- OCR quality depends on image clarity, layout, handwriting, and model availability.
- Always verify names, dates, amounts, and legal clauses against the source document.
- Do not use the output as legal advice or as an authoritative contract record.